In [ ]:
!python -m spacy download fr_core_news_md
!python -m spacy download en_core_web_md

In [ ]:
import pandas as pd
import datetime
import os
import json
from io import StringIO
import numpy as np
import plotly.express as px
import re
import spacy



In [ ]:
import s3fs #pour connecter au bucket

In [ ]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



In [ ]:
with fs.open(f"{BUCKET_OUT}/Data_bso/outputs/enriched_data/geolocation/2026-03-11_publi_from_bso_with_address.csv", "r") as file_in:
    df0 = pd.read_csv(file_in, sep=",")

file_path = BUCKET_OUT + "/" + "Data_bso/fichier_from_mesr/bso-publications-latest_199318270_enriched.csv"
with fs.open(file_path) as csvin:
    df1 = pd.read_csv(csvin, sep =";")


file_path = BUCKET_OUT + "/" + "Data_bso/outputs/enriched_data/geolocation/2026-03-11_affil_crossref.csv"
with fs.open(file_path) as csvin:
    dfcr = pd.read_csv(csvin, sep =",")

dfcr1 = dfcr.loc[dfcr.affiliation!="no affil"]
df01 = df0.merge(dfcr1[["id","affiliation"]], on = ["id"], how = "left").merge(df1[["id","title"]], on = ["id"], how = "left")#.reset_index()
df01

In [ ]:
df02 = df01[["id","struc_address","affiliation","title"]]
df02["address"] = df02.struc_address.str.split("|")

df_ad = df02[['id','title','address']].explode("address").drop_duplicates().dropna(subset="address").reset_index().drop(columns=["index"])
df_ad["rank_id"] = df_ad.id.astype(str) +"_"+ df_ad.index.astype(str)
#On isole les adresses françaises
df_other = df_ad.loc[~(df_ad.address.str.lower().str.contains("france|paris")) & (df_ad.address != "no address")].reset_index().drop(columns=["index"])

df_other

In [ ]:
import requests
from requests.exceptions import HTTPError
import urllib.parse
import tqdm

ROR_API_ENDPOINT = "https://api.dev.ror.org/organizations"


params = {"query":"1m84wm78"} # le nombre de ligne. Par défaut, l'api affiche les 30 première ligne et le maximum fixé est de 10000.

In [ ]:
doc = nlp("University of Southampton")
for ent in doc.ents:
    #print(ent.text)
    #ent.
    if ent.label_ in ["ORG","GPE","LOC"]:
        print(ent.text, ent.label_)
        search_univ = re.findall(r"(?:univ\w*|coll[èe]ge|institute*|école|school)", ent.text.lower())
        print(len(search_univ))
    
    else:
        pass

In [ ]:
def request_ror_api(query):
    response = requests.get(ROR_API_ENDPOINT + '?' + f'query="{query.lower()}"').json()
    try:
        if response["number_of_results"] > 1 :
            item = response["items"][0]
            #print(item)
            if 'education' in item["types"] and 'funder' in item["types"] :
                rors = item["id"]
            else:
                for n, x in enumerate(response["items"]):
                    if 'education' in x["types"] and 'funder' in x["types"] :
                        datas = response["items"][n]
                        rors = datas["id"]
                    else:
                        rors = "not found"
        elif response["number_of_results"] == 1 :
            datas = response["items"][0]
            rors = datas["id"]
        else:
            rors = "not found"
    except:
        rors = "not found"

    return rors
    

In [ ]:
nlp = spacy.load("en_core_web_md")

In [ ]:
dict_ror = {}

for index, row in tqdm.tqdm(df_other.iterrows(), total = len(df_other)):
    #print("######", row.address)
    clean_address= re.sub(r"\s-\s", ". ", row.address)
    doc = nlp(clean_address)
    list_ror = []
    for ent in doc.ents:
        #print(ent.text)
 
        search_univ = re.findall(r"(?:university|coll[èe]ge|institute*|école|school)", ent.text.lower())


            
        if len(search_univ) > 0:
            #print(ent.text, ent.label_)
            rors = request_ror_api(ent.text)
            if rors != "not found":
                list_ror.append(rors)
    if len(list_ror) > 0:
        dict_ror[row.rank_id] = "|".join(list(set([ror for ror in list_ror])))
            

dict_ror

In [ ]:
df_other["rors"] = df_other.rank_id.map(dict_ror.get)
df_other.loc[~df_other.rors.isna()]

In [ ]:
nlp = spacy.load("fr_core_news_md")

In [ ]:

for index, row in tqdm.tqdm(df_other.loc[df_other.rors.isna()].iterrows(), total = len(df_other.loc[df_other.rors.isna()])):
    #print("######", row.address)
    clean_address= re.sub(r"\s-\s", ". ", row.address)
    doc = nlp(clean_address)
    list_ror = []
    for ent in doc.ents:
        #print(ent.text)
 
        search_univ = re.findall(r"(?:université|coll[èe]ge|institut|école|school|centre|laboratoire)", ent.text.lower())


            
        if len(search_univ) > 0:
            #print(ent.text, ent.label_)
            rors = request_ror_api(ent.text)
            if rors != "not found":
                list_ror.append(rors)
    if len(list_ror) > 0:
        dict_ror[row.rank_id] = "|".join(list(set([ror for ror in list_ror])))
            

dict_ror

In [ ]:
df_other["rors"] = df_other.rank_id.map(dict_ror.get)
df_other

In [ ]:
dict_ror["halhal-04095085_339"]
df_other.loc[df_other.rors.isna()]

In [ ]:
response = requests.get(ROR_API_ENDPOINT + '?' + f'query="université laval"').json()
request_ror_api("université laval")